# S2 — Prototype Python : EBP et enchères

Projet 3 — Théorie des jeux et systèmes multi-agents. Exécuter depuis le dossier `notebooks/`.

## 1. Jeux de signalisation et EBP

La nature tire un type $t$ selon $P(t)$. L’émetteur choisit un message $m$, le récepteur observe $m$ et choisit une action $a$. Sur le chemin d’équilibre, $\mu(t\mid m)=P(t)\sigma(m\mid t)/\sum_{t’}P(t’)\sigma(m\mid t’)$. Hors chemin, Bayes ne détermine pas la croyance. Un EBP exige la rationalité séquentielle des deux joueurs et la cohérence des croyances sur le chemin.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
from jeux import equilibres_ebp, enchere
import pandas as pd
import numpy as np

### Exemple didactique (non Bière ou Quiche)

Ce jeu jouet permet de vérifier le mécanisme du solveur sans prétendre reproduire les gains exacts du cours. Les deux joueurs souhaitent que l’action révèle correctement le type.

In [ ]:
def gain_emetteur(t, m, a):
    return float((t == "H" and a == "accepter") or (t == "L" and a == "refuser"))

def gain_recepteur(t, m, a):
    return gain_emetteur(t, m, a)

equilibres = equilibres_ebp(["H", "L"], [0.6, 0.4], ["x", "y"], ["accepter", "refuser"], gain_emetteur, gain_recepteur, grille=0.1)
print("Nombre de profils EBP sur la grille :", len(equilibres))
print("Exemple :", equilibres[0])

### Bière ou Quiche : modèle classique pédagogique (Cho–Kreps)

**Origine des gains : hypothèse explicite**, car les tables numériques ne figurent pas dans les deux chapitres transmis. Nous utilisons une normalisation classique compatible avec les résultats du sujet : l'émetteur reçoit 2 si le récepteur ne combat pas, plus 1 pour son petit-déjeuner préféré (Fort → Bière, Faible → Quiche). Le récepteur préfère combattre le Faible et ne pas combattre le Fort (gain 1 si la décision correspond au type, 0 sinon). Ces gains ne sont **pas présentés comme les gains officiels du cours**.

| Type | Message | Combattre (émetteur, récepteur) | Ne pas combattre (émetteur, récepteur) |
|---|---|---|---|
| Fort | Bière | (1, 0) | (3, 1) |
| Fort | Quiche | (0, 0) | (2, 1) |
| Faible | Bière | (0, 1) | (2, 0) |
| Faible | Quiche | (1, 1) | (3, 0) |

Le récepteur combat si $\mu(\mathrm{Faible}\mid m)>0{,}5$ ; à $0{,}5$, il est indifférent. Les croyances sur le chemin se calculent avec Bayes ; hors chemin, elles sont libres mais doivent rendre l'action optimale.


In [ ]:
from jeux import equilibria_biere_quiche, decision_recepteur

for p_fort in (0.9, 0.3):
    eq = equilibria_biere_quiche(p_fort=p_fort)
    print(f"P(Fort)={p_fort} : {len(eq)} EBP sur la grille")
    for profil in eq:
        if profil["nature"] == "poolant":
            print("  ", profil)

print("Seuil du récepteur :", [(p, decision_recepteur(p)) for p in (0.2, 0.5, 0.51, 0.6)])

### Vérification du pooling Bière et du critère intuitif

À $P(\mathrm{Fort})=0{,}9$, si les deux types choisissent **Bière**, Bayes impose $P(\mathrm{Faible}\mid\mathrm{Bière})=0{,}1$ : le récepteur ne combat pas. Pour **Quiche** (hors chemin), la croyance $0{,}6$ conduit à combattre et dissuade les deux types de dévier ; la croyance $0{,}2$ conduit à ne pas combattre et permet au Faible de dévier (gain 3 au lieu de 2).

Le pooling **Quiche** peut aussi être soutenu, pour $P(\mathrm{Fort})=0{,}9$, par une croyance hors chemin après Bière qui amène le récepteur à combattre. Le **critère intuitif** l'écarte dans ce modèle : après une déviation vers Bière, le Faible ne peut pas améliorer son gain de 3, tandis que le Fort peut passer de 2 à 3 si le récepteur ne combat pas. La déviation signale donc le Fort, auquel cas le récepteur ne combat pas et le Fort dévie effectivement.

À $P(\mathrm{Fort})=0{,}3$, les conclusions changent : le récepteur combat sur le chemin d'un pooling car $P(\mathrm{Faible})=0{,}7$. **Ne pas transposer les résultats de $0{,}9$ sans vérifier les meilleures réponses.**


## 2. Enchères scellées à valeurs privées

Hypothèses : $n$ joueurs symétriques, valeurs i.i.d. uniformes sur $[0,1]$, neutralité au risque. Premier prix : $b(v)=\frac{n-1}{n}v$. Second prix : $b(v)=v$. À l’équilibre, la recette attendue est $\frac{n-1}{n+1}$. Le paramètre `alpha` définit la stratégie du joueur 0, les autres restent à l’équilibre.

In [ ]:
lignes=[]
for n in [2,3,5]:
    for prix in ["premier", "second"]:
        alpha=(n-1)/n if prix=="premier" else 1.0
        r=enchere(n,alpha,N=100_000,seed=42,prix=prix)
        lignes.append({"n":n,"format":prix,"recette simulée":r["recette_moyenne"],"recette théorique":r["recette_theorique_equilibre"],"IC 95% bas":r["ic95_recette"][0],"IC 95% haut":r["ic95_recette"][1]})
pd.DataFrame(lignes)

### Déviations unilatérales

Faire varier $\alpha$ pour le joueur 0 tandis que ses adversaires gardent la stratégie d’équilibre. Les gains simulés varient légèrement à cause de Monte-Carlo. Un maximum empirique proche de $\frac{n-1}{n}$ est attendu au premier prix.

In [ ]:
n=3
alphas=np.linspace(0.4,0.9,11)
resultats=[(round(float(a),3),enchere(n,float(a),N=100_000,seed=42)["gain_moyen_joueur_0"]) for a in alphas]
pd.DataFrame(resultats,columns=["alpha","gain moyen joueur 0"])

## 3. Interprétation et limites

Le solveur énumère des stratégies **pures** et discrétise les croyances hors chemin : il ne couvre pas les stratégies mixtes ni toutes les croyances continues. Pour le jeu Bière ou Quiche, les gains exacts du cours sont indispensables. Les intervalles de confiance sont des IC normaux Monte-Carlo à 95 % pour la recette. L’intégration des outils LangChain est prévue en S3.

## 4. Analyse structurée des déviations unilatérales

On compare la stratégie théorique aux autres coefficients $\alpha$ pour un seul joueur, tandis que les adversaires suivent leur stratégie d’équilibre. Les petites différences Monte-Carlo ne sont pas des preuves mathématiques.


In [ ]:
from jeux import analyser_deviations
for n in [2, 3, 5]:
    for prix in ["premier", "second"]:
        alpha_eq = (n-1)/n if prix == "premier" else 1.0
        grille = sorted(set(np.linspace(0, 1, 21).tolist() + [alpha_eq]))
        analyse = analyser_deviations(n, grille, N=100_000, seed=42, prix=prix)
        tableau = pd.DataFrame(analyse["deviations"])
        meilleur = tableau.loc[tableau["gain_moyen"].idxmax()]
        print(f"n={n}, {prix}: alpha équilibre={alpha_eq:.4f}, meilleur alpha simulé={meilleur.alpha:.4f}, écart={meilleur.ecart_vs_equilibre:.5f}")
        display(tableau)
